<h1 style="font-size:34px; text-align:center; color: green; font-weight:bold; margin-bottom:5px;">
  HERBAL STORE ANALYTICS
</h1>

<h2 style="font-size:24px; text-align: center; font-weight:bold; color: #34A99D;">
  DATA_CLEANING
</h2>

In [37]:
import pandas as pd
import numpy as np
import os
import warnings
import logging
from datetime import datetime
warnings.filterwarnings('ignore')

## **1. LOAD DATA**

In [38]:
orders = pd.read_excel("../data/order_raw.xlsx")

In [39]:
customers = pd.read_excel("../data/customers_raw.xlsx")

In [40]:
products = pd.read_excel("../data/products_raw.xlsx")

## **2. DATA INSPECTION**

In [41]:
orders.head()

,order id,order date,variant sku,product qty,subtotal price,discount,Tax,total price,customer id
0,100,2025-11-01 01:45:23,HERB127,3,1350,27.00,67.5,1390.50,85
1,101,2025-11-01 02:19:55,HERB174,2,624,12.48,31.2,642.72,488
2,102,2025-11-01 02:54:27,HERB002,2,360,7.20,18.0,370.80,111
3,103,2025-11-01 03:28:59,HERB130,4,720,14.40,36.0,741.60,507
4,104,2025-11-01 04:03:31,HERB010,3,642,12.84,32.1,661.26,1080


In [42]:
orders.dtypes

order id                   int64
order date        datetime64[us]
variant sku                  str
product qty                int64
subtotal price             int64
discount                 float64
Tax                      float64
total price              float64
customer id                int64
dtype: object

In [43]:
customers.head()

,Customer ID,First Name,Last Name,FullName,Phone Number,Email ID,City,State,Country,Postal Code
0,1,Aarav,Seshad,Aarav Seshad,606853615,AaravSeshad60@gmail.com,Pune,Maharashtra,India,51522
1,2,Abhay,Sodhi,Abhay Sodhi,2779010432,AbhaySodhi27@gmail.com,Chandigarh,Chandigarh,India,143410
2,3,Amogh,Shankar,Amogh Shankar,911798089324,AmoghShankar91@gmail.com,Hyderabad,Telangana,India,95396
3,4,Armaan,Viswanathan,Armaan Viswanathan,1888880670,ArmaanViswanathan18@gmail.com,Shimla,Himachal Pradesh,India,540515
4,5,Balaji,Natarajan,Balaji Natarajan,910585277221,BalajiNatarajan91@gmail.com,Vadodara,Gujarat,India,43030


In [44]:
products.head()

,Variant SKU,Variant Price,Cost per item,Variant Inventory Qty,True SKU margins,Product Name,Category
0,HERB001,250,160.0,4235,90.0,Amla Hair Oil,Hair care
1,HERB002,180,110.0,8760,70.0,Neem Face Wash,Skin care
2,HERB003,300,190.0,2644,110.0,Tulsi Green Tea,Teas
3,HERB004,90,50.0,6488,40.0,Aloe Vera Gel,Skin care
4,HERB005,220,140.0,6416,80.0,Ashwagandha Capsules,CapsulesTablets


In [45]:
products.dtypes

Variant SKU                  str
Variant Price              int64
Cost per item            float64
Variant Inventory Qty      int64
True SKU margins         float64
Product Name                 str
Category                     str
dtype: object

In [46]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 14596 entries, 0 to 14595
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order id        14596 non-null  int64         
 1   order date      14596 non-null  datetime64[us]
 2   variant sku     14596 non-null  str           
 3   product qty     14596 non-null  int64         
 4   subtotal price  14596 non-null  int64         
 5   discount        14596 non-null  float64       
 6   Tax             14596 non-null  float64       
 7   total price     14596 non-null  float64       
 8   customer id     14596 non-null  int64         
dtypes: datetime64[us](1), float64(3), int64(4), str(1)
memory usage: 1.1 MB


Have 14,596 records(orders) over 9 fields.\
No missing values.\
Datatypes of all the fields are matched correct.

In [47]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Variant SKU            176 non-null    str    
 1   Variant Price          176 non-null    int64  
 2   Cost per item          176 non-null    float64
 3   Variant Inventory Qty  176 non-null    int64  
 4   True SKU margins       176 non-null    float64
 5   Product Name           176 non-null    str    
 6   Category               176 non-null    str    
dtypes: float64(2), int64(2), str(3)
memory usage: 15.8 KB


Have 176 records(products) over 7 fields.\
No missing values.\
Datatypes of all the fields are matched correct.

### **Check duplicates**

In [48]:
orders.duplicated().sum()

np.int64(0)

In [49]:
products.duplicated().sum()

np.int64(0)

In [50]:
customers.duplicated().sum()

np.int64(0)

No duplicated records

### **Standardise column names**

In [51]:
def clean_col(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

In [52]:
orders = clean_col(orders)
customers = clean_col(customers)
products = clean_col(products)

In [53]:
orders.columns

Index(['order_id', 'order_date', 'variant_sku', 'product_qty',
       'subtotal_price', 'discount', 'tax', 'total_price', 'customer_id'],
      dtype='str')

## **MERGE DATA**

In [54]:
df = orders.merge(customers, on="customer_id", how="left")
df = df.merge(products, on="variant_sku", how="left")

In [55]:
df.head()

,order_id,order_date,variant_sku,product_qty,subtotal_price,discount,tax,total_price,customer_id,first_name,...,city,state,country,postal_code,variant_price,cost_per_item,variant_inventory_qty,true_sku_margins,product_name,category
0,100,2025-11-01 01:45:23,HERB127,3,1350,27.00,67.5,1390.50,85,Naman,...,Surat,Gujarat,India,670662,450,360.0,5702,90.0,Amla Herbal Conditioner,Hair care
1,101,2025-11-01 02:19:55,HERB174,2,624,12.48,31.2,642.72,488,Hitesh,...,Solapur,Maharashtra,India,413001,312,249.6,5783,62.4,Sandalwood Herbal Soap,Skin care
2,102,2025-11-01 02:54:27,HERB002,2,360,7.20,18.0,370.80,111,Hamsa,...,Mysuru,Karnataka,India,569725,180,110.0,8760,70.0,Neem Face Wash,Skin care
3,103,2025-11-01 03:28:59,HERB130,4,720,14.40,36.0,741.60,507,Yashdeep,...,Saharanpur,Uttar Pradesh,India,247001,180,144.0,5812,36.0,Aloe Vera Night Cream,Skin care
4,104,2025-11-01 04:03:31,HERB010,3,642,12.84,32.1,661.26,1080,Arsh,...,Ajmer,Rajasthan,India,305001,214,171.2,3049,42.8,Giloy Juice,Wellness


In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14596 entries, 0 to 14595
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   order_id               14596 non-null  int64         
 1   order_date             14596 non-null  datetime64[us]
 2   variant_sku            14596 non-null  str           
 3   product_qty            14596 non-null  int64         
 4   subtotal_price         14596 non-null  int64         
 5   discount               14596 non-null  float64       
 6   tax                    14596 non-null  float64       
 7   total_price            14596 non-null  float64       
 8   customer_id            14596 non-null  int64         
 9   first_name             14596 non-null  str           
 10  last_name              14596 non-null  str           
 11  fullname               14596 non-null  str           
 12  phone_number           14596 non-null  int64         
 13  email_id    

In [57]:
df.to_csv("../data/sales.csv", index=False)